In [14]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import the production pipeline tools
from KneeBot import KneeBotDataBuilder, KneeBotModelTrainer

In [15]:
# Data Preparation
# 1. Initialize the builder (sets state only)
builder = KneeBotDataBuilder('Final Results - OA Study.csv')

# 2. Execute deferred processing to clean, build targets, and split the data
X_train, X_test, y_train, y_test, target_names = builder.prepare_and_export_data()

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Number of targets: {len(target_names)}")

Training features shape: (1352, 123)
Testing features shape: (338, 123)
Number of targets: 31


In [16]:
# Logistic Regression Model for Comparison
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, hamming_loss, classification_report

class KneeBotLogisticTrainer:
    """
    Manages the lifecycle of the KneeBot Logistic Regression model.
    Wraps a base LogisticRegression model in a MultiOutputClassifier to handle
    the multi-label OTC and Exercise targets, mirroring the KneeBotModelTrainer API.
    """

    def __init__(self):
        """
        Initializes the Multi-Label Logistic Regression classifier.
        Sets up a state variable to hold the dynamic target column names.
        """
        # max_iter=1000: Ensures convergence on highly dimensional or complex preprocessed data.
        # random_state=42: Ensures reproducible training results.
        base_lr = LogisticRegression(max_iter=1000, random_state=42)

        # MultiOutputClassifier fits one classifier per target label.
        # n_jobs=-1: Utilizes all available CPU cores to accelerate training across multiple labels.
        self.model = MultiOutputClassifier(base_lr, n_jobs=-1)

        # State variable to store the exact names of the target columns learned from the builder
        self.target_names_ = None

    def train(self, X_train_transformed, y_train, target_names: list):
        """
        Fits the Multi-Label Logistic Regression model to the fully preprocessed training data.

        Args:
            X_train_transformed: The preprocessed feature matrix (DataFrame or Numpy array).
            y_train: The multi-label target matrix (DataFrame).
            target_names: A list of the target column names from the data builder.
        """
        # Store the target names as a class attribute for evaluation and export
        self.target_names_ = target_names

        print("Training the Multi-Label Logistic Regression model...")
        self.model.fit(X_train_transformed, y_train)
        print("Model training complete.\n")

        return self

    def evaluate(self, X_test_transformed, y_test):
        """
        Generates predictions on the unseen test set and calculates key multi-label metrics.

        Args:
            X_test_transformed: The preprocessed test feature matrix.
            y_test: The true targets for the test set.
        """
        if self.target_names_ is None:
            raise ValueError("The model must be trained before evaluation. Call train() first.")

        # Generate predictions
        y_pred = self.model.predict(X_test_transformed)

        # 1. Exact Match Ratio (Strict Accuracy)
        # Calculates the percentage of patients where EVERY single label was predicted perfectly.
        exact_match = accuracy_score(y_test, y_pred)
        print(f"Exact Match Ratio (Strict Accuracy): {exact_match:.4f}")

        # 2. Hamming Loss
        # Calculates the fraction of individual labels (across the entire matrix) that are incorrect.
        h_loss = hamming_loss(y_test, y_pred)
        print(f"Hamming Loss: {h_loss:.4f} (meaning {h_loss * 100:.2f}% of single predictions were wrong)")

        # 3. Classification Report
        # Provides Precision, Recall, and F1-Score for each specific class.
        print("\n--- Classification Report ---")
        # zero_division=0 prevents warnings if a rare class receives 0 correct predictions
        print(classification_report(
            y_test,
            y_pred,
            target_names=self.target_names_,
            zero_division=0
        ))

    def export_production_assets(self):
        """
        Serializes and saves the trained model and the dynamic target names to disk.
        Files are prefixed with 'kneebot_lr_' to avoid overwriting Random Forest assets.
        """
        if self.target_names_ is None:
            raise ValueError("No model or target names found. You must train the model before exporting.")

        # Save the fully trained Multi-Label Logistic Regression model
        model_filename = 'kneebot_lr_model.joblib'
        joblib.dump(self.model, model_filename)
        print(f"Saved Logistic Regression Model to '{model_filename}'")

        # Save the list of target names
        targets_filename = 'kneebot_lr_target_names.joblib'
        joblib.dump(self.target_names_, targets_filename)
        print(f"Saved Target Names to '{targets_filename}'")

In [17]:
# K-Nearest Neighbors (KNN) Model for Comparison
import pandas as pd
import numpy as np
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, hamming_loss, classification_report

class KneeBotKNNTrainer:
    """
    Manages the lifecycle of the KneeBot K-Nearest Neighbors (KNN) model.
    Handles training, performance evaluation, and exporting the final production assets
    for offline comparison against the baseline Random Forest model.
    """

    def __init__(self):
        """
        Initializes the KNN multi-label classifier.
        Sets up a state variable to hold the dynamic target column names.
        """
        # n_neighbors=5: Standard default for local neighborhood voting.
        # n_jobs=-1: Utilizes all available CPU cores to accelerate distance calculations.
        # Note: KNeighborsClassifier natively supports multi-label outputs without wrappers.
        self.model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

        # State variable to store the exact names of the target columns learned from the builder
        self.target_names_ = None

    def train(self, X_train_transformed, y_train, target_names: list):
        """
        Fits the KNN model to the fully preprocessed training data.

        Args:
            X_train_transformed: The preprocessed feature matrix (DataFrame or Numpy array).
            y_train: The multi-label target matrix (DataFrame).
            target_names: A list of the target column names from the data builder.
        """
        # Store the target names as a class attribute so they can be used in evaluation and exported
        self.target_names_ = target_names

        print("Training the Multi-Label KNN model...")
        self.model.fit(X_train_transformed, y_train)
        print("Model training complete.\n")

        return self

    def evaluate(self, X_test_transformed, y_test):
        """
        Generates predictions on the unseen test set and calculates key multi-label metrics.

        Args:
            X_test_transformed: The preprocessed test feature matrix.
            y_test: The true targets for the test set.
        """
        if self.target_names_ is None:
            raise ValueError("The model must be trained before evaluation. Call train() first.")

        # Generate predictions
        y_pred = self.model.predict(X_test_transformed)

        # 1. Exact Match Ratio (Strict Accuracy)
        # Calculates the percentage of patients where EVERY single label was predicted perfectly.
        exact_match = accuracy_score(y_test, y_pred)
        print(f"Exact Match Ratio (Strict Accuracy): {exact_match:.4f}")

        # 2. Hamming Loss
        # Calculates the fraction of individual labels (across the entire matrix) that are incorrect.
        h_loss = hamming_loss(y_test, y_pred)
        print(f"Hamming Loss: {h_loss:.4f} (meaning {h_loss * 100:.2f}% of single predictions were wrong)")

        # 3. Classification Report
        # Provides Precision, Recall, and F1-Score for each of the specific OTC and Exercise classes.
        print("\n--- Classification Report ---")
        # zero_division=0 prevents warnings if a rare class receives 0 correct predictions
        print(classification_report(
            y_test,
            y_pred,
            target_names=self.target_names_,
            zero_division=0
        ))

    def export_production_assets(self):
        """
        Serializes and saves the trained model and the dynamic target names to disk.
        Files are prefixed with 'kneebot_knn_' to avoid overwriting Random Forest assets.
        """
        if self.target_names_ is None:
            raise ValueError("No model or target names found. You must train the model before exporting.")

        # Save the fully trained KNN model
        model_filename = 'kneebot_knn_model.joblib'
        joblib.dump(self.model, model_filename)
        print(f"Saved KNN Model to '{model_filename}'")

        # Save the list of target names
        targets_filename = 'kneebot_knn_target_names.joblib'
        joblib.dump(self.target_names_, targets_filename)
        print(f"Saved Target Names to '{targets_filename}'")

In [18]:
print("=============================================")
print(" MODEL 1: PRODUCTION RANDOM FOREST")
print("=============================================\n")

# Instantiate existing production model
rf_trainer = KneeBotModelTrainer()
rf_trainer.train(X_train, y_train, target_names)
rf_trainer.evaluate(X_test, y_test)

print("\n=============================================")
print(" MODEL 2: LOGISTIC REGRESSION")
print("=============================================\n")

# Instantiate the new LR comparison model
lr_trainer = KneeBotLogisticTrainer()
lr_trainer.train(X_train, y_train, target_names)
lr_trainer.evaluate(X_test, y_test)

print("\n=============================================")
print(" MODEL 3: K-Nearest Neighbors (KNN)")
print("=============================================\n")

# Instantiate the new KNN comparison model
knn_trainer = KneeBotKNNTrainer()
knn_trainer.train(X_train, y_train, target_names)
knn_trainer.evaluate(X_test, y_test)

 MODEL 1: PRODUCTION RANDOM FOREST

Training the Multi-Label Random Forest model...
Model training complete.

Exact Match Ratio (Strict Accuracy): 0.0148
Hamming Loss: 0.2100 (meaning 21.00% of single predictions were wrong)

--- Classification Report ---
                                                  precision    recall  f1-score   support

                Target_OTC_Acetaminophen_Tylenol       0.57      0.66      0.61       194
                            Target_OTC_Ibuprofen       0.53      0.47      0.50       165
                             Target_OTC_Ice_pack       0.53      0.20      0.30       132
                          Target_OTC_Heating_pad       0.58      0.16      0.26       128
                         Target_OTC_Voltaren_Gel       0.28      0.08      0.13       111
                             Target_OTC_Naproxen       0.33      0.05      0.09        98
                              Target_OTC_Icy_Hot       0.56      0.06      0.10        90
                       